This notebook process the netcdf files in `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion` and `data/SWOT_L3_LR_SSH_Expert_v3_0` one cycle at a time.

Steps are the following:
- retrieve all the cycles from the netcdf file names formatted as `SWOT_L3_LR_SSH_Expert_{cycle_num}_{pass_num}_{fromdate}T{fromtime}_{todate}T{totime}.nc` (same formatting for both input directories),
- for each cycle it:
    - checks if the pass 584 is present in `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion`, if not it skipped the cycle as it is not complete,
    - creates the file name `SWOT_L3_LR_SSH_Expert_{cycle_num}.png`,
    - checks if the file name is already present in the directory `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion_plots`,
    - if it is not present, it:
        - opens the netcdf files of the cycle using `xr.open_mfdatasets`,
        - computes the difference between the cyclogeostrophic (minimization-based) currents speed and the geostrophic currents speed,
        - creates a plot representing this difference (using `cmocean.balance` colormap, and `cartopy.crs.PlateCarre` projection/transformation),
        - saves it as a HD png image the directory `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion_plots`.

In [ ]:
import glob
import os
import re
import time
import warnings

import cartopy.crs as ccrs
import cmocean.cm as cmo
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [ ]:
warnings.filterwarnings("ignore")

INPUT_CG_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion"
INPUT_G_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0"
OUTPUT_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion_plots"

os.makedirs(OUTPUT_DIR, exist_ok=True)

pattern = re.compile(r"SWOT_L3_LR_SSH_Expert_(\d+)_(\d+)_\d+T\d+_\d+T\d+_v3\.0\.nc")


def add_pass_dim(ds, varnames):
    m = pattern.match(os.path.basename(ds.encoding["source"]))
    pass_num = int(m.group(2))
    ds = ds[varnames]
    ds = ds.where(np.abs(ds.latitude) > 10)
    return ds.expand_dims(pass_num=[pass_num])


while True:
    cg_files = sorted(glob.glob(os.path.join(INPUT_CG_DIR, "*.nc")))

    cycles = {}
    for f in cg_files:
        m = pattern.match(os.path.basename(f))
        if m:
            cycle_num, pass_num = m.group(1), m.group(2)
            cycles.setdefault(cycle_num, set()).add(pass_num)

    for cycle_num, passes in sorted(cycles.items()):
        out_filename = f"SWOT_L3_LR_SSH_Expert_{cycle_num}.png"
        out_path = os.path.join(OUTPUT_DIR, out_filename)

        if os.path.exists(out_path):
            print(f"Plot for cycle {cycle_num} already exists, skipping")
            continue

        try:
            cycle_cg_files = sorted(glob.glob(os.path.join(INPUT_CG_DIR, f"SWOT_L3_LR_SSH_Expert_{cycle_num}_*.nc")))
            cycle_g_files = sorted(glob.glob(os.path.join(INPUT_G_DIR, f"SWOT_L3_LR_SSH_Expert_{cycle_num}_*.nc")))

            ds_cg = xr.open_mfdataset(
                cycle_cg_files, preprocess=lambda ds: add_pass_dim(ds, ["ucg_mb", "vcg_mb"])
            )
            ds_g = xr.open_mfdataset(
                cycle_g_files, preprocess=lambda ds: add_pass_dim(ds, ["ugos_filtered", "vgos_filtered"])
            )

            speed_g = (ds_g.ugos_filtered ** 2 + ds_g.vgos_filtered ** 2) ** 0.5
            speed_cg = (ds_cg.ucg_mb ** 2 + ds_cg.vcg_mb ** 2) ** 0.5
            speed_diff = speed_cg - speed_g

            fig, ax = plt.subplots(figsize=(23, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))
            for p in speed_diff.pass_num.values:
                sd = speed_diff.sel(pass_num=p)
                ax.pcolormesh(
                    sd.longitude.values, sd.latitude.values, sd.values,
                    cmap=cmo.balance, vmin=-0.5, vmax=0.5,
                    transform=ccrs.PlateCarree(),
                )
            # Add colorbar from the last pcolormesh
            sm = plt.cm.ScalarMappable(cmap=cmo.balance, norm=plt.Normalize(vmin=-0.5, vmax=0.5))
            cbar = fig.colorbar(
                sm, ax=ax, extend="both", label="$\\| \mathbf{u}_{cg} \\| - \\| \mathbf{u}_g \\| (\\text{m s}^{-1})$"
            )
            cbar.set_label("$\\| \mathbf{u}_{cg} \\| - \\| \mathbf{u}_g \\| (\\text{m s}^{-1})$", fontsize=15)
            cbar.ax.tick_params(labelsize=13)
            ax.coastlines()
            ax.set_title(f"Cycle {cycle_num}", fontsize=16)
            fig.savefig(out_path, dpi=300, bbox_inches="tight")
            plt.close(fig)

            ds_cg.close()
            ds_g.close()
        except Exception:
            print(f"Error for cycle {cycle_num}")
    
    time.sleep(1)